# 02 — EDA Escalade : Climb Dataset (jordizar)

**Objectif** : Explorer le dataset escalade pour comprendre la distribution des profils grimpeurs, identifier les variables prédictives du niveau, et préparer la modélisation.

**Dataset** : jordizar/climb-dataset — 10 927 grimpeurs uniques  
**Source** : Kaggle (données issues de 8a.nu)  
**Colonnes clés** : sex, height, weight, age, years_cl, grades_max

---

## Plan
1. Chargement et audit qualité
2. Table de conversion des grades
3. Distributions physiologiques (taille, poids, âge, années pratique)
4. Distribution des niveaux (grades_max)
5. Corrélations variables physiologiques → niveau
6. Analyse par sous-groupes (sexe, années de pratique)
7. Détection et traitement des outliers
8. Conclusions et variables retenues pour la modélisation

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)

## 1. Chargement et audit qualité

In [ ]:
df = pd.read_csv('../data/raw/climbing/climber_df.csv')
df_grades = pd.read_csv('../data/raw/climbing/grades_conversion_table.csv')

print(f"Dataset grimpeurs : {len(df):,} lignes, {len(df.columns)} colonnes")
print(f"Table grades : {len(df_grades)} lignes")
print()
df.head()

In [ ]:
print("=== Types ===")
print(df.dtypes)
print()
print("=== Valeurs manquantes ===")
print(df.isnull().sum())
print()
print("=== Statistiques descriptives ===")
print(df[['height', 'weight', 'age', 'years_cl', 'grades_max', 'grades_mean']].describe().round(2))

## 2. Table de conversion des grades

In [ ]:
# Nettoyage de la table de conversion
df_grades = df_grades[['grade_id', 'grade_fra']].copy()
df_grades = df_grades[df_grades['grade_fra'] != '-'].reset_index(drop=True)

# Créer un mapping id → cotation
grade_map = dict(zip(df_grades['grade_id'], df_grades['grade_fra']))

# Ajouter la cotation française au dataset principal
df['grade_fra'] = df['grades_max'].map(grade_map)

print("Plage de grades dans le dataset :")
print(f"  Min : {df['grades_max'].min()} → {grade_map.get(df['grades_max'].min(), '?')}")
print(f"  Max : {df['grades_max'].max()} → {grade_map.get(df['grades_max'].max(), '?')}")
print(f"  Médiane : {df['grades_max'].median():.0f} → {grade_map.get(int(df['grades_max'].median()), '?')}")
print()

# Afficher les 20 grades les plus fréquents
top_grades = df['grade_fra'].value_counts().head(20)
print("Top 20 grades les plus fréquents :")
print(top_grades.to_string())

## 3. Distributions physiologiques

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

vars_config = [
    ('height', 'Taille (cm)', 'steelblue'),
    ('weight', 'Poids (kg)', 'teal'),
    ('age', 'Âge', 'coral'),
    ('years_cl', 'Années de pratique', 'mediumpurple'),
    ('grades_max', 'Grade max (numérique)', 'goldenrod'),
    ('grades_count', 'Nombre de voies réalisées', 'gray'),
]

for ax, (col, label, color) in zip(axes.flat, vars_config):
    data = df[col].dropna()
    ax.hist(data, bins=40, color=color, edgecolor='white', alpha=0.85)
    ax.axvline(data.median(), color='red', linestyle='--', linewidth=1.5, label=f'Médiane : {data.median():.1f}')
    ax.set_title(label)
    ax.set_xlabel(label)
    ax.set_ylabel('Fréquence')
    ax.legend(fontsize=9)

plt.suptitle('Distributions physiologiques — Dataset escalade', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig('../data/processed/eda_climbing_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Distribution des niveaux

In [ ]:
# Regrouper par tranche de niveau pour lisibilité
bins = list(range(29, 78, 3))
labels_bin = [grade_map.get(b, str(b)) for b in bins[:-1]]
df['grade_bin'] = pd.cut(df['grades_max'], bins=bins, labels=labels_bin)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Distribution brute
axes[0].hist(df['grades_max'], bins=40, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution du grade max (numérique)')
axes[0].set_xlabel('Grade max')
axes[0].set_ylabel('Nombre de grimpeurs')

# Ajouter les labels cotation sur l'axe x
key_grades = {37: '6a', 41: '6b', 45: '6c', 49: '7a', 53: '7b', 57: '7c', 61: '8a', 65: '8b'}
axes[0].set_xticks(list(key_grades.keys()))
axes[0].set_xticklabels(list(key_grades.values()), rotation=45)

# Distribution par sexe
for sex, color, label in [(0, 'steelblue', 'Hommes'), (1, 'salmon', 'Femmes')]:
    subset = df[df['sex'] == sex]['grades_max']
    axes[1].hist(subset, bins=30, alpha=0.6, color=color, edgecolor='white',
                 label=f'{label} (n={len(subset):,}, médiane={subset.median():.0f})')

axes[1].set_title('Distribution grade max par sexe')
axes[1].set_xlabel('Grade max')
axes[1].set_ylabel('Nombre de grimpeurs')
axes[1].set_xticks(list(key_grades.keys()))
axes[1].set_xticklabels(list(key_grades.values()), rotation=45)
axes[1].legend()

plt.tight_layout()
plt.savefig('../data/processed/eda_climbing_grades.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nBiais dataset identifié :")
print(f"  Grade médian : {df['grades_max'].median():.0f} ≈ {grade_map.get(int(df['grades_max'].median()), '?')}")
print("  → Dataset surreprésenté en grimpeurs avancés (8a.nu = plateforme de grimpeurs sérieux)")
print("  → À documenter dans les limites du modèle")

## 5. Corrélations variables → niveau

In [ ]:
# BMI
df['bmi'] = df['weight'] / (df['height'] / 100) ** 2

feature_cols = ['sex', 'height', 'weight', 'bmi', 'age', 'years_cl', 'grades_count']
target_col = 'grades_max'

corr = df[feature_cols + [target_col]].corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Bar chart des corrélations
colors = ['steelblue' if v > 0 else 'salmon' for v in corr.values]
axes[0].barh(corr.index, corr.values, color=colors)
axes[0].axvline(0, color='black', linewidth=0.8)
axes[0].set_title(f'Corrélation des variables avec le grade max')
axes[0].set_xlabel('Corrélation de Pearson')
for i, v in enumerate(corr.values):
    axes[0].text(v + 0.005 * (1 if v >= 0 else -1), i, f'{v:.3f}', va='center', fontsize=9)

# Scatter years_cl vs grades_max (variable la plus prédictive attendue)
sample = df.sample(min(2000, len(df)), random_state=42)
axes[1].scatter(sample['years_cl'], sample['grades_max'], alpha=0.2, s=8, color='steelblue')
axes[1].set_title('Années de pratique vs Grade max')
axes[1].set_xlabel('Années de pratique')
axes[1].set_ylabel('Grade max')

# Ajouter labels cotation sur y
axes[1].set_yticks(list(key_grades.keys()))
axes[1].set_yticklabels(list(key_grades.values()))

plt.tight_layout()
plt.savefig('../data/processed/eda_climbing_correlations.png', dpi=150, bbox_inches='tight')
plt.show()

print("\nCorrélations avec grades_max :")
print(corr.to_string())

## 6. Analyse par sous-groupes

In [ ]:
df['years_group'] = pd.cut(df['years_cl'], bins=[0, 2, 5, 10, 20, 50],
                            labels=['0-2 ans', '3-5 ans', '6-10 ans', '11-20 ans', '20+ ans'])

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Grade médian par années de pratique
grade_by_years = df.groupby('years_group')['grades_max'].median()
grade_labels = [grade_map.get(int(v), str(int(v))) for v in grade_by_years.values]

bars = axes[0].bar(grade_by_years.index, grade_by_years.values, color='steelblue')
axes[0].set_title('Grade médian par expérience')
axes[0].set_xlabel('Années de pratique')
axes[0].set_ylabel('Grade max médian')
axes[0].set_yticks(list(key_grades.keys()))
axes[0].set_yticklabels(list(key_grades.values()))
for bar, label in zip(bars, grade_labels):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, label,
                 ha='center', va='bottom', fontsize=9)

# Evolution grade avec l'âge
df['age_group'] = pd.cut(df['age'], bins=[10, 20, 30, 40, 50, 60, 80],
                          labels=['<20', '20-30', '30-40', '40-50', '50-60', '60+'])

grade_by_age = df.groupby(['age_group', 'sex'])['grades_max'].median().unstack()
grade_by_age.columns = ['Hommes', 'Femmes']
grade_by_age.plot(kind='bar', ax=axes[1], color=['steelblue', 'salmon'], width=0.7)
axes[1].set_title('Grade médian par groupe d\'âge et sexe')
axes[1].set_xlabel('Groupe d\'âge')
axes[1].set_ylabel('Grade max médian')
axes[1].set_yticks(list(key_grades.keys()))
axes[1].set_yticklabels(list(key_grades.values()))
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.savefig('../data/processed/eda_climbing_subgroups.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Détection et traitement des outliers

In [ ]:
print("=== Valeurs aberrantes potentielles ===")

checks = [
    ('height', 140, 220, 'cm'),
    ('weight', 40, 130, 'kg'),
    ('age', 10, 80, 'ans'),
    ('years_cl', 0, 50, 'ans'),
    ('bmi', 15, 40, 'kg/m²'),
]

total_outliers = 0
for col, low, high, unit in checks:
    outliers = df[(df[col] < low) | (df[col] > high)]
    total_outliers += len(outliers)
    print(f"  {col} [{low}-{high} {unit}] : {len(outliers)} outliers ({100*len(outliers)/len(df):.1f}%)")

print(f"\nTotal outliers : {total_outliers}")

# Filtrage
df_clean = df[
    (df['height'].between(140, 220)) &
    (df['weight'].between(40, 130)) &
    (df['age'].between(10, 80)) &
    (df['years_cl'].between(0, 50)) &
    (df['bmi'].between(15, 40))
].copy()

print(f"\nAprès filtrage : {len(df_clean):,} lignes ({100*len(df_clean)/len(df):.1f}% conservé)")

## 8. Export dataset nettoyé

In [ ]:
export_cols = ['user_id', 'sex', 'height', 'weight', 'bmi', 'age', 'years_cl',
               'grades_max', 'grades_mean', 'grades_count', 'grade_fra']

df_export = df_clean[export_cols].copy()
df_export.to_csv('../data/processed/climbing_clean.csv', index=False)
print(f"Dataset nettoyé exporté : {len(df_export):,} lignes")
print(f"Fichier : data/processed/climbing_clean.csv")
print()
print(df_export.describe().round(2))

## 9. Conclusions EDA Escalade

### Variables retenues pour la modélisation
| Variable | Impact prédit | Justification |
|---|---|---|
| `years_cl` | **Très fort** | Variable la plus prédictive — apprentissage progressif |
| `sex` | Fort | Différence de niveau médian significative H/F |
| `weight` | Modéré | Rapport force/poids déterminant en escalade |
| `bmi` | Modéré | Synthèse taille/poids |
| `height` | Faible | Corrélation faible isolément |
| `age` | Faible-modéré | Influence via expérience et physique |

### Variable dead hang (non disponible dans ce dataset)
- Le dataset ne contient pas de test de suspension
- En mode avancé dans l'app, cette variable sera collectée auprès de l'utilisateur
- Elle sera utilisée comme feature supplémentaire si le modèle le permet, sinon intégrée via une règle métier

### Biais documenté
- Grade médian ≈ 7b — dataset orienté grimpeurs avancés
- Prédictions moins fiables pour les débutants (< 5b)
- Ce biais est assumé et communiqué dans l'interface

### Cible de prédiction
- **Target** : `grades_max` (numérique) → affiché en cotation française via `grade_map`
- **Intervalle de confiance** : ±1 grade (défendable sur ce dataset)